In [1]:
# List of Hurricane Evacuation Centers in New York City with Addresses
# Each item is a tuple with the name of the center and its address
locations = [
    ('Norman Thomas HS (ECF)', '111 E 33rd St, NYC, New York'),
    ('Midtown East Campus', '233 E 56th St, NYC, New York'),
    ('Louis D. Brandeis HS', '145 W 84th St, NYC, New York'),
    ('Martin Luther King, Jr. HS', '122 Amsterdam Avenue, NYC, New York'),
    ('P.S. 48', '4360 Broadway, NYC, New York')
]

In [2]:
from geopy.geocoders import Nominatim

locations_with_coordinates = []
geolocator = Nominatim(user_agent="assignment1_geocoder")
for name, address in locations:
    location = geolocator.geocode(address)
    print(f"{name}: {location.latitude}, {location.longitude}")
    locations_with_coordinates.append((name, location.latitude, location.longitude))

Norman Thomas HS (ECF): 40.7462177, -73.9809816
Midtown East Campus: 40.6513247, -73.9242165
Louis D. Brandeis HS: 40.7860963, -73.9742272
Martin Luther King, Jr. HS: 40.7747751, -73.9853689
P.S. 48: 40.8530757, -73.933713


In [3]:
import folium

# 1. Calcola il centro della mappa facendo la media delle coordinate
avg_lat = sum(loc[1] for loc in locations_with_coordinates) / len(locations_with_coordinates)
avg_lon = sum(loc[2] for loc in locations_with_coordinates) / len(locations_with_coordinates)

# 2. Inizializza la mappa interattiva centrata sui tuoi punti
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=10)

# 3. Aggiungi un marker per ogni posizione
for name, lat, lon in locations_with_coordinates:
    folium.Marker(
        location=[lat, lon],
        tooltip=name,          # Compare quando passi il mouse sopra il marker
        popup=name,            # Facoltativo: compare quando clicchi sul marker
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(m)

# 4. Mostra la mappa nel notebook
m

In [4]:
import requests
import folium
from IPython.display import display

# chiave API di OpenRouteService
API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImJmM2IyMTQ0ZTY4ODQzYTc4OGNhNTQ5YzExZTIwMjQ3IiwiaCI6Im11cm11cjY0In0="

# URL dell'endpoint di ottimizzazione
url = 'https://api.openrouteservice.org/optimization'

headers = {
    'Authorization': API_KEY,
    'Content-Type': 'application/json'
}

# Punto di partenza/ritorno: primo elemento di locations_with_coordinates
start_name   = locations_with_coordinates[0][0]
start_coords = [locations_with_coordinates[0][2], locations_with_coordinates[0][1]]  # [lon, lat]

if len(locations_with_coordinates) >= 2:

    # Definizione del problema di ottimizzazione
    payload = {
        "vehicles": [
            {
                "id": 1,
                "profile": "driving-car",
                "start": start_coords,
                "end":   start_coords
            }
        ],
        "jobs": [
            {
                "id": i + 1,
                "description": locations_with_coordinates[i][0],
                "location": [locations_with_coordinates[i][2], locations_with_coordinates[i][1]]
            }
            for i in range(1, len(locations_with_coordinates))   # salta il deposito (indice 0)
        ],
        "options": {
        "g": True  # richiede la geometria del percorso
    }
    }

    # Invio della richiesta POST
    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        data = response.json()
        print("Ottimizzazione completata con successo!\n")

        route = data['routes'][0]

        print(f"Durata totale stimata: {route['duration'] / 60:.1f} minuti")

        print("\nOrdine ottimale delle tappe:")
        print("-> Partenza dal Deposito")
        for step in route['steps']:
            if step['type'] == 'job':
                print(f"-> Vai alla Tappa con ID: {step['id']} (Arrivo stimato al secondo: {step['arrival']})")
        print("-> Ritorno al Deposito")

        # CREAZIONE DELLA MAPPA FOLIUM
        map_center = [start_coords[1], start_coords[0]]   # [lat, lon] per Folium
        m = folium.Map(location=map_center, zoom_start=13)

        # Marker del Deposito (Verde)
        folium.Marker(
            location=[start_coords[1], start_coords[0]],
            popup=start_name,
            tooltip=f"START/END: {start_name}",
            icon=folium.Icon(color="green", icon="home")
        ).add_to(m)

        # Marker dei Clienti (Blu) con numero d'ordine
        ordine_visita = 1
        for step in route['steps']:
            if step['type'] == 'job':
                job_id       = step['id']
                nome_tappa = next(j['description'] for j in payload['jobs'] if j['id'] == job_id)
                step_lon, step_lat = step['location']

                folium.Marker(
                    location=[step_lat, step_lon],
                    popup=nome_tappa,
                    tooltip=f"Fermata {ordine_visita}: {nome_tappa}",
                    icon=folium.Icon(color="blue", icon="info-sign")
                ).add_to(m)
                ordine_visita += 1

        # Tracciamento del percorso reale
        if 'geometry' in route:
            import polyline as pl
            coords_strada = pl.decode(route['geometry'])  # [(lat, lon), ...]
            folium.PolyLine(
                locations=coords_strada,
                color="#ef4444",
                weight=5,
                opacity=0.8,
                tooltip="Percorso Ottimizzato"
            ).add_to(m)

        print("\nDone! Visualizzazione mappa:")
        display(m)

    else:
        print(f"Errore API ORS {response.status_code}: {response.text}")

else:
    print("Errore: Punti insufficienti.")

Ottimizzazione completata con successo!

Durata totale stimata: 100.0 minuti

Ordine ottimale delle tappe:
-> Partenza dal Deposito
-> Vai alla Tappa con ID: 4 (Arrivo stimato al secondo: 535)
-> Vai alla Tappa con ID: 3 (Arrivo stimato al secondo: 810)
-> Vai alla Tappa con ID: 5 (Arrivo stimato al secondo: 1635)
-> Vai alla Tappa con ID: 2 (Arrivo stimato al secondo: 4275)
-> Ritorno al Deposito

Done! Visualizzazione mappa:
